# 07 - Results Interpretation

**Purpose:** consolidate the final results around the project question:

> Can time-series history, social-network exposure, and review-language signals help forecast short-term shifts in community attention toward local Yelp businesses?

In [1]:
from pathlib import Path
import json

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "new_orleans"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

METRICS_OUTPUT_PATH = OUTPUTS_DIR / "forecasting_metrics.csv"
PREDICTIONS_OUTPUT_PATH = OUTPUTS_DIR / "forecasting_predictions.csv"
PULSE_METRICS_OUTPUT_PATH = OUTPUTS_DIR / "attention_pulse_metrics.csv"
PULSE_PREDICTIONS_OUTPUT_PATH = OUTPUTS_DIR / "attention_pulse_predictions.csv"
PULSE_TOPK_OUTPUT_PATH = OUTPUTS_DIR / "attention_pulse_topk_metrics.csv"
GRAPH_SUMMARY_PATH = PROCESSED_DIR / "social_graph_summary.json"
FEATURE_SUMMARY_PATH = PROCESSED_DIR / "forecasting_feature_summary.json"

metrics = pd.read_csv(METRICS_OUTPUT_PATH)
predictions = pd.read_csv(PREDICTIONS_OUTPUT_PATH)
pulse_metrics = pd.read_csv(PULSE_METRICS_OUTPUT_PATH)
pulse_predictions = pd.read_csv(PULSE_PREDICTIONS_OUTPUT_PATH)
pulse_topk_metrics = pd.read_csv(PULSE_TOPK_OUTPUT_PATH)
with GRAPH_SUMMARY_PATH.open("r", encoding="utf-8") as file:
    graph_summary = json.load(file)
with FEATURE_SUMMARY_PATH.open("r", encoding="utf-8") as file:
    feature_summary = json.load(file)

metrics.sort_values(["split", "WAPE", "MAE"])

,split,task,model,train_period,validation_period,test_period,rows,MAE,RMSE,WAPE
0,primary_covid_test,review_count_regression,Baseline: last month,2015-02 to 2018-12,2019-01 to 2019-12,2020-01 to 2021-12,21024,1.601170,3.007150,0.664292
1,primary_covid_test,review_count_regression,Baseline: rolling 3-month avg,2015-02 to 2018-12,2019-01 to 2019-12,2020-01 to 2021-12,21024,1.646071,3.364436,0.682921
2,primary_covid_test,review_count_regression,ML: historical + SNA,2015-02 to 2018-12,2019-01 to 2019-12,2020-01 to 2021-12,21024,1.970705,3.525411,0.817604
3,primary_covid_test,review_count_regression,ML: historical + NLP,2015-02 to 2018-12,2019-01 to 2019-12,2020-01 to 2021-12,21024,1.996907,3.570527,0.828475
4,primary_covid_test,review_count_regression,ML: historical + business,2015-02 to 2018-12,2019-01 to 2019-12,2020-01 to 2021-12,21024,1.997847,3.636709,0.828865
5,primary_covid_test,review_count_regression,ML: historical,2015-02 to 2018-12,2019-01 to 2019-12,2020-01 to 2021-12,21024,2.004549,3.607005,0.831646
6,primary_covid_test,review_count_regression,ML: all modalities,2015-02 to 2018-12,2019-01 to 2019-12,2020-01 to 2021-12,21024,2.034262,3.608341,0.843973
7,primary_covid_test,review_count_regression,Baseline: seasonal naive,2015-02 to 2018-12,2019-01 to 2019-12,2020-01 to 2021-12,21024,3.596128,7.490338,1.491959
8,secondary_pre_covid_test,review_count_regression,ML: all modalities,2015-02 to 2017-12,2018-01 to 2018-12,2019-01 to 2019-12,10511,2.302558,3.989596,0.381137
9,secondary_pre_covid_test,review_count_regression,ML: historical + business,2015-02 to 2017-12,2018-01 to 2018-12,2019-01 to 2019-12,10511,2.307195,4.082504,0.381904


## Regression Interpretation

Summarize which feature group performs best in each time split and whether the all-modality model improves on simpler alternatives.


In [2]:
regression_summary_rows = []
for split_name, split_metrics in metrics.groupby("split"):
    ranked = split_metrics.sort_values("WAPE").reset_index(drop=True)
    best = ranked.iloc[0]
    hist = split_metrics[split_metrics["model"] == "ML: historical"].iloc[0]
    business = split_metrics[split_metrics["model"] == "ML: historical + business"].iloc[0]
    sna = split_metrics[split_metrics["model"] == "ML: historical + SNA"].iloc[0]
    nlp = split_metrics[split_metrics["model"] == "ML: historical + NLP"].iloc[0]
    all_modalities = split_metrics[split_metrics["model"] == "ML: all modalities"].iloc[0]
    regression_summary_rows.append({
        "split": split_name,
        "best_model": best["model"],
        "best_WAPE": best["WAPE"],
        "historical_WAPE": hist["WAPE"],
        "business_WAPE": business["WAPE"],
        "sna_WAPE": sna["WAPE"],
        "nlp_WAPE": nlp["WAPE"],
        "all_modalities_WAPE": all_modalities["WAPE"],
        "all_vs_historical_relative_change": (all_modalities["WAPE"] - hist["WAPE"]) / hist["WAPE"],
        "all_vs_business_relative_change": (all_modalities["WAPE"] - business["WAPE"]) / business["WAPE"],
    })
regression_summary = pd.DataFrame(regression_summary_rows)
regression_summary

,split,best_model,best_WAPE,historical_WAPE,business_WAPE,sna_WAPE,nlp_WAPE,all_modalities_WAPE,all_vs_historical_relative_change,all_vs_business_relative_change
0,primary_covid_test,Baseline: last month,0.664292,0.831646,0.828865,0.817604,0.828475,0.843973,0.014823,0.018227
1,secondary_pre_covid_test,ML: all modalities,0.381137,0.388882,0.381904,0.388126,0.385845,0.381137,-0.019915,-0.002010


## Pulse Interpretation

Summarize pulse-classification performance with class balance in mind. The evaluation now separates thresholded detection from top-k retrieval:

- F1 uses a cutoff tuned on the validation period;
- precision@k and recall@k show whether the model can prioritize likely attention pulses for analyst review.

In [3]:
# Pulse summaries emphasize rare-event detection rather than overall accuracy.
pulse_summary_rows = []
for split_name, split_metrics in pulse_metrics.groupby("split"):
    ranked = split_metrics.sort_values(["F1", "PR_AUC"], ascending=[False, False]).reset_index(drop=True)
    best = ranked.iloc[0]
    hist = split_metrics[split_metrics["model"] == "ML: historical"].iloc[0]
    business = split_metrics[split_metrics["model"] == "ML: historical + business"].iloc[0]
    sna = split_metrics[split_metrics["model"] == "ML: historical + SNA"].iloc[0]
    nlp = split_metrics[split_metrics["model"] == "ML: historical + NLP"].iloc[0]
    all_modalities = split_metrics[split_metrics["model"] == "ML: all modalities"].iloc[0]

    split_topk_10 = pulse_topk_metrics[
        (pulse_topk_metrics["split"] == split_name)
        & (pulse_topk_metrics["k_fraction"] == 0.10)
    ].copy()
    best_topk_10 = split_topk_10.sort_values(["precision_at_k", "recall_at_k"], ascending=[False, False]).iloc[0]
    all_topk_10 = split_topk_10[split_topk_10["model"] == "ML: all modalities"].iloc[0]

    pulse_summary_rows.append({
        "split": split_name,
        "positive_rate": all_modalities["positive_rate"],
        "best_model": best["model"],
        "best_F1": best["F1"],
        "best_PR_AUC": best["PR_AUC"],
        "best_threshold": best["decision_threshold"],
        "best_validation_F1": best["validation_F1"],
        "historical_F1": hist["F1"],
        "business_F1": business["F1"],
        "sna_F1": sna["F1"],
        "nlp_F1": nlp["F1"],
        "all_modalities_F1": all_modalities["F1"],
        "all_modalities_PR_AUC": all_modalities["PR_AUC"],
        "all_modalities_threshold": all_modalities["decision_threshold"],
        "best_precision_at_10pct_model": best_topk_10["model"],
        "best_precision_at_10pct": best_topk_10["precision_at_k"],
        "best_recall_at_10pct": best_topk_10["recall_at_k"],
        "all_modalities_precision_at_10pct": all_topk_10["precision_at_k"],
        "all_modalities_recall_at_10pct": all_topk_10["recall_at_k"],
    })
pulse_summary = pd.DataFrame(pulse_summary_rows)
pulse_summary

,split,positive_rate,best_model,best_F1,best_PR_AUC,best_threshold,best_validation_F1,historical_F1,business_F1,sna_F1,nlp_F1,all_modalities_F1,all_modalities_PR_AUC,all_modalities_threshold,best_precision_at_10pct_model,best_precision_at_10pct,best_recall_at_10pct,all_modalities_precision_at_10pct,all_modalities_recall_at_10pct
0,primary_covid_test,0.102407,Baseline: rising recent activity,0.280882,0.148129,0.500,0.215558,0.218521,0.235885,0.220746,0.219755,0.243177,0.184456,0.400,Baseline: previous-month pulse rule,0.240133,0.234556,0.225392,0.220158
1,secondary_pre_covid_test,0.131957,ML: historical + NLP,0.293900,0.237426,0.435,0.359316,0.281658,0.290868,0.283972,0.293900,0.282132,0.227054,0.425,ML: historical,0.301331,0.228551,0.283270,0.214852


In [4]:
print("Social graph summary")
for key, value in graph_summary.items():
    print(f"{key}: {value}")

print("\nForecasting dataset summary")
for key, value in feature_summary.items():
    print(f"{key}: {value}")

Social graph summary
active_review_threshold: 5
threshold_candidates: [2, 3, 5, 10, 20]
edge_weight_formula: 1 + log1p(shared_business_count) + category_jaccard
reviewing_users: 245421
matched_user_profiles: 245419
active_users: 26598
graph_nodes: 26598
graph_edges: 116558
mean_edge_weight: 1.833072733525831
mean_edge_shared_business_count: 2.048550936014688
mean_edge_category_jaccard: 0.2671618532172656
connected_components: 11508
largest_component_size: 14965
isolated_active_users: 11387
community_method: weighted_louvain_largest_component
communities_assigned: 63
threshold_sensitivity_output: C:\Users\mehdi\OneDrive\Documents\community-forecasting-yelp\data\processed\new_orleans\active_reviewer_threshold_sensitivity.csv
output: C:\Users\mehdi\OneDrive\Documents\community-forecasting-yelp\data\processed\new_orleans\user_network_features.csv

Forecasting dataset summary
min_total_reviews: 100
min_active_months: 36
business_count: 876
row_count: 68249
feature_month_min: 2015-01
feature

In [5]:
for _, row in regression_summary.iterrows():
    split = row["split"]
    all_vs_hist = row["all_vs_historical_relative_change"] * 100
    all_vs_business = row["all_vs_business_relative_change"] * 100
    print(f"{split} regression:")
    print(f"  Best model: {row['best_model']} with WAPE={row['best_WAPE']:.4f}")
    print(f"  All modalities WAPE: {row['all_modalities_WAPE']:.4f}")
    print(f"  All modalities vs historical: {all_vs_hist:+.2f}%")
    print(f"  All modalities vs historical+business: {all_vs_business:+.2f}%")

print()
for _, row in pulse_summary.iterrows():
    split = row["split"]
    print(f"{split} attention pulses:")
    print(f"  Positive rate: {row['positive_rate']:.3f}")
    print(f"  Best thresholded model: {row['best_model']} with F1={row['best_F1']:.4f}, PR-AUC={row['best_PR_AUC']:.4f}")
    print(f"  Best model threshold: {row['best_threshold']:.3f}; validation F1={row['best_validation_F1']:.4f}")
    print(f"  All modalities F1: {row['all_modalities_F1']:.4f}, PR-AUC={row['all_modalities_PR_AUC']:.4f}, threshold={row['all_modalities_threshold']:.3f}")
    print(f"  Best precision@10%: {row['best_precision_at_10pct_model']} with precision={row['best_precision_at_10pct']:.4f}, recall={row['best_recall_at_10pct']:.4f}")

primary_covid_test regression:
  Best model: Baseline: last month with WAPE=0.6643
  All modalities WAPE: 0.8440
  All modalities vs historical: +1.48%
  All modalities vs historical+business: +1.82%
secondary_pre_covid_test regression:
  Best model: ML: all modalities with WAPE=0.3811
  All modalities WAPE: 0.3811
  All modalities vs historical: -1.99%
  All modalities vs historical+business: -0.20%

primary_covid_test attention pulses:
  Positive rate: 0.102
  Best thresholded model: Baseline: rising recent activity with F1=0.2809, PR-AUC=0.1481
  Best model threshold: 0.500; validation F1=0.2156
  All modalities F1: 0.2432, PR-AUC=0.1845, threshold=0.400
  Best precision@10%: Baseline: previous-month pulse rule with precision=0.2401, recall=0.2346
secondary_pre_covid_test attention pulses:
  Positive rate: 0.132
  Best thresholded model: ML: historical + NLP with F1=0.2939, PR-AUC=0.2374
  Best model threshold: 0.435; validation F1=0.3593
  All modalities F1: 0.2821, PR-AUC=0.2271, 

## Interpretation Structure

1. **Time series:** recent review patterns remain the strongest reference point for raw review-count forecasting.
2. **Validation discipline:** pulse models now tune their decision threshold on a validation year before final test evaluation.
3. **Business metadata:** static context helps most in the pre-COVID split, where business popularity and category context are more stable.
4. **SNA:** social exposure is measurable and reportable, but it does not consistently improve prediction after temporal and business signals are included.
5. **NLP:** lightweight review-language features add an extra modality, but their incremental gain is modest.
6. **Target design:** attention pulses are better evaluated as both classification and ranking. Top-k metrics are useful because analysts may only inspect the highest-risk business-months.

## Final Position

The improved evaluation makes the final story more credible: model thresholds are no longer chosen on the test set or fixed arbitrarily at `0.5`. The test results now answer two separate questions: whether a model can classify pulses at a tuned cutoff, and whether it can rank the most likely attention shifts near the top of the list.

The main empirical result remains mixed but useful. Simple temporal baselines are still strong under COVID-era disruption, while ML models help more in the pre-COVID split and can be assessed more fairly with validation-tuned thresholds and top-k metrics. SNA and NLP are valuable for interpretation and modality coverage, but they provide limited incremental predictive lift in this dataset.

Key limitations:

- Yelp friendship links are static.
- SNA features measure exposure, not causal influence.
- NLP features are lightweight lexicon/text-length signals.
- COVID-era disruption changes predictability.
- Review activity is a proxy for Yelp attention, not revenue or true customer volume.
- Static business metadata may include end-of-dataset information.